# Activity 3 — Correcting a wrong answer

This activity shows how the instructions can make the assistant contradict the policy. A wrong answer like this is often called a hallucination. Run the cells from top to bottom in a new kernel. The first run is intentionally wrong; inspect its activity log before correcting it.

## Purpose
Find an answer that contradicts the policy and correct the instructions given to the assistant.

## What to expect
The first instructions say Gold members get two checked bags; the policy found says one. The first answer follows the instructions and contradicts the policy. The corrected instructions follow the policy and answer one. The activity-log names are `activity-3-hallucination-broken` and `activity-3-hallucination-fixed`.

## Bad practice
Tell the assistant a fact that the policy it found does not support.

## Good practice
Tell the assistant to follow the policy it found, even when that contradicts its assumptions.

## Why the good practice works
The activity log shows both the instructions and the policy. If the instructions defer to the policy, the answer stays consistent with the policy.

## Deliverable
Both activity-log links, the contradiction you found, and the instruction change.


In [1]:
from IPython.display import Markdown, display
from experiment_utils import langfuse, print_summary, run_hallucination, show_trace


In [2]:
broken_prompt = "You are Skyline Travel's assistant. Gold members are always entitled to two checked bags. Answer using the retrieved policy."
broken = run_hallucination(broken_prompt, "broken")
print("answer:", broken["answer"])
show_trace(broken["trace_id"])


answer: As a Gold member, you’re entitled to **two checked bags**.

The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you.
activity log: http://localhost:3000/project/langfuse-observability/traces/744d5434377b504abcf91ea7043bc2da


'http://localhost:3000/project/langfuse-observability/traces/744d5434377b504abcf91ea7043bc2da'

## What went wrong?

Open the first activity log and compare the instructions with the policy the assistant found. The policy says one checked bag; the instructions say two. Find the contradiction and correct the instructions.


In [3]:
fixed_prompt = "You are Skyline Travel's assistant. Answer using only the retrieved policy documents. If they conflict with your assumptions, follow the documents."
fixed = run_hallucination(fixed_prompt, "fixed")
print("answer:", fixed["answer"])
show_trace(fixed["trace_id"])
rows = ["| run | policy found | answer | input tokens | output tokens | time (s) | activity log |",
        "|---|---|---|---|---|---|---|"]
for run in [broken, fixed]:
    doc = run["doc"]
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['label']} | {doc['title']} ({doc['tier']}) | {answer} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | {run['trace_url']} |")
display(Markdown("\n".join(rows)))


answer: Your Gold fare includes **one 23 kg checked bag only**. A second checked bag costs **60 EUR**.
activity log: http://localhost:3000/project/langfuse-observability/traces/d05f5e757d20ff4388428cff16804bfc


| run | policy found | answer | input tokens | output tokens | time (s) | activity log |
|---|---|---|---|---|---|---|
| before fix | Gold fare baggage (gold) | As a Gold member, you’re entitled to **two checked bags**.  The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you. | 95 | 332 | 4.8 | http://localhost:3000/project/langfuse-observability/traces/744d5434377b504abcf91ea7043bc2da |
| after fix | Gold fare baggage (gold) | Your Gold fare includes **one 23 kg checked bag only**. A second checked bag costs **60 EUR**. | 98 | 55 | 1.0 | http://localhost:3000/project/langfuse-observability/traces/d05f5e757d20ff4388428cff16804bfc |

## Wrap-up

Run this cell to print the session and activity-log links, then send the remaining activity logs to Langfuse. Submit the links and your findings.


In [4]:
print_summary(3)
langfuse.flush()
print("activity logs sent")

session activity-3-10317e02
activity-3-broken: http://localhost:3000/project/langfuse-observability/traces/744d5434377b504abcf91ea7043bc2da
activity-3-fixed: http://localhost:3000/project/langfuse-observability/traces/d05f5e757d20ff4388428cff16804bfc
activity logs sent
